# LlamaIndex와 AgentCore 메모리 - 법률 문서 분석기 (장기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 **장기 메모리**를 활용한 법률 문서 분석기를 만드는 방법을 보여줍니다. 여러 사건과 법적 절차에 걸쳐 지속적으로 법률 지식을 축적하고, 수개월에서 수년에 걸친 판례를 추적할 수 있습니다.

## 아키텍처 개요

![LlamaIndex AgentCore 장기 메모리 아키텍처](LlamaIndex-AgentCore-LTM-Arch.png)

## 튜토리얼 세부 정보

**튜토리얼 세부 정보:**
- **튜토리얼 유형**: 장기 크로스 세션 메모리
- **에이전트 사용 사례**: 법률 문서 분석기
- **에이전트 프레임워크**: LlamaIndex
- **LLM 모델**: Anthropic Claude 3.7 Sonnet
- **튜토리얼 구성 요소**: AgentCore 장기 메모리, LlamaIndex 에이전트, 법률 도구
- **예제 복잡도**: 고급

## 비즈니스 가치

**기업 법률 인텔리전스**: 사건 지식을 축적하고, 법률 전략 진화를 추적하며, 사건과 기간에 걸쳐 포괄적인 판례 데이터베이스를 유지하는 지속적 AI 메모리로 법률 업무를 혁신하세요.

**주요 전문적 이점:**
- **사건 연속성**: 법률 사안 및 팀원 간 원활한 지식 이전
- **판례 데이터베이스**: 중요한 판례법, 전략 및 결과를 영구적으로 보존
- **크로스 케이스 인텔리전스**: 여러 법률 사안에 걸친 패턴과 연관성 파악
- **전략적 우위**: 과거 사건 데이터를 활용한 우수한 법률 포지셔닝
- **고객 가치**: 다년간 고객 관계를 위한 상세한 컨텍스트 유지
- **리스크 관리**: 규제 변화와 법률 전략에 미치는 영향 추적

## 장기 메모리 구성

**기술 설정**: 이 튜토리얼은 12개월 보존을 위한 시맨틱 전략과 함께 AgentCore 메모리를 사용합니다:
- **메모리 유형**: 자동 인사이트 추출을 위한 시맨틱 전략
- **보존 기간**: 법률 사건 연속성을 위한 365일 이벤트 만료
- **크로스 세션**: 동일 actor_id + memory_id, 법률 기간별 다른 session_id
- **검색 기능**: 사건 이력 전반에 걸친 시맨틱 검색을 위한 내장 메모리 검색 도구

## 기술 개요

**주요 장기 메모리 구성 요소:**
1. **시맨틱 전략 구성**: 365일 보존의 자동 인사이트 추출을 위한 SemanticStrategy 사용
2. **크로스 세션 지속성**: 동일 actor_id + memory_id, 기간별 다른 session_id로 지식 연속성 확보
3. **커스텀 메모리 검색 도구**: AgentCore의 네이티브 search_long_term_memories()를 LlamaIndex FunctionTool로 래핑
4. **시맨틱 처리 파이프라인**: 대화 이벤트 → 시맨틱 메모리 변환을 위한 90초 대기
5. **동적 세션 관리**: 유연한 세션 처리를 위한 memory.context.session_id 사용

**학습 내용:**

- 여러 법률 사건에 걸친 지속적 AgentCore 메모리 생성
- 시간에 따른 누적 법률 지식 구축
- 판례법 및 판례 전반에 걸친 시맨틱 검색 구현
- 법률 전략 진화 및 사건 결과 추적
- 크로스 세션 법률 지식 지속성 및 검색 테스트

## 시나리오 컨텍스트

이 예제에서는 수개월에서 수년에 걸친 여러 사건에서 법률 지식을 유지하는 "법률 문서 분석기"를 만듭니다. 분석기는 AgentCore 메모리를 사용하여 판례, 법률 전략, 규제 변화, 사건 결과에 대한 지속적 지식 베이스를 구축하며, 이는 시간이 지남에 따라 성장하고 진화하여 정교한 종합적 법률 분석을 가능하게 합니다.

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 종속성 설치 및 설정

In [ ]:
# 시맨틱 전략 툴킷을 포함한 필수 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3 bedrock-agentcore-starter-toolkit

In [ ]:
# 필수 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies.semantic import SemanticStrategy
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

print("모든 종속성을 성공적으로 임포트했습니다!")

## 2단계: AgentCore 메모리 구성

장기 법률 지식을 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# 장기 지속성을 위한 시맨틱 전략으로 AgentCore 메모리 생성
region = os.getenv('AWS_REGION', 'us-east-1')
memory_manager = MemoryManager(region_name=region)

try:
    # 자동 인사이트 추출을 위한 시맨틱 전략으로 메모리 생성
    memory = memory_manager.get_or_create_memory(
        name=f'LegalAnalyzerSemantic_{int(datetime.now().timestamp())}',
        strategies=[SemanticStrategy(name="legalLongTermMemory")],
        event_expiry_days=365  # 법률 기록을 위한 12개월 보존
    )
    memory_id = memory.get('id')
    print(f"시맨틱 메모리 생성 완료: {memory_id}")
    print(f"   상태: {memory.get('status')}")
    print(f"   전략: {[s.get('name') if isinstance(s, dict) else str(s) for s in memory.get('strategies', [])]}")
    
    # 메모리가 ACTIVE 상태가 될 때까지 대기
    if memory.get('status') != 'ACTIVE':
        print(f"\n메모리가 ACTIVE 상태가 될 때까지 대기 중 (현재 {memory.get('status')})...")
        import time
        max_wait = 300  # 최대 5분
        waited = 0
        while waited < max_wait:
            time.sleep(10)
            waited += 10
            # 상태 확인
            current_memory = memory_manager.get_memory(memory_id)
            status = current_memory.get('status')
            print(f"   [{waited}초] 상태: {status}")
            if status == 'ACTIVE':
                print(f"메모리가 ACTIVE 상태가 되었습니다! ({waited}초 소요)")
                break
        else:
            print(f"경고: {max_wait}초 후에도 메모리가 ACTIVE 상태가 아닙니다. 계속 진행합니다...")
    
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 법률 도구 구현

종합적 법률 분석을 위한 전문 도구를 정의합니다:

In [ ]:
def analyze_contract_clause(case_id: str, clause_type: str, risk_level: str, precedent_reference: str) -> str:
    """Analyze contract clause with risk assessment and precedent reference"""
    return f"계약 조항 분석 완료: {case_id}의 {clause_type} (리스크: {risk_level})"

def track_case_precedent(case_id: str, precedent_case: str, legal_principle: str, applicability: str) -> str:
    """Track case precedent with legal principle and applicability assessment"""
    return f"판례 추적: {case_id} - {precedent_case} - {legal_principle}"

def document_regulatory_change(regulation_type: str, change_description: str, impact_assessment: str, client_implications: str) -> str:
    """Document regulatory change with client portfolio implications"""
    print(f"규제 업데이트: {regulation_type} - {change_description} (영향: {impact_assessment})")
    return f"규제 변경 기록 완료: {regulation_type}"

def update_legal_strategy(case_id: str, strategy_type: str, approach: str, confidence_level: str) -> str:
    """Update legal strategy for specific case"""
    print(f"법률 전략: {case_id} - {strategy_type} ({confidence_level} 확신도)")
    return f"전략 업데이트 완료: {case_id}"

def log_case_outcome(case_id: str, outcome_type: str, result: str, lessons_learned: str) -> str:
    """Log case outcome with lessons learned"""
    print(f"사건 결과: {case_id} - {outcome_type}: {result}")
    return f"결과 기록 완료: {case_id}"

def log_legal_milestone(quarter: str, milestone: str, details: str) -> str:
    """Log a legal milestone with quarter and detailed progress"""
    print(f"법률 마일스톤 - {quarter}: {milestone}")
    return f"{quarter} 마일스톤 기록 완료: {milestone} - {details}"

def track_legal_metrics(metric_type: str, value: str, case_id: str, quarter: str) -> str:
    """Track specific legal metrics with case and timeline"""
    print(f"{quarter}: {metric_type} = {value} ({case_id} 대상)")
    return f"{metric_type} 추적 완료: {case_id}의 {quarter} - {value}"

def save_legal_insight(insight: str, quarter: str, legal_context: str) -> str:
    """Save legal insights with context"""
    print(f"{quarter} 인사이트: {insight[:50]}...")
    return f"{quarter} 인사이트 저장 완료 (법률 컨텍스트: {legal_context})"

# 에이전트를 위한 도구 객체 생성
legal_tools = [
    FunctionTool.from_defaults(fn=analyze_contract_clause),
    FunctionTool.from_defaults(fn=track_case_precedent),
    FunctionTool.from_defaults(fn=document_regulatory_change),
    FunctionTool.from_defaults(fn=update_legal_strategy),
    FunctionTool.from_defaults(fn=log_case_outcome),
    FunctionTool.from_defaults(fn=log_legal_milestone),
    FunctionTool.from_defaults(fn=track_legal_metrics),
    FunctionTool.from_defaults(fn=save_legal_insight)
]

print("법률 도구 생성 완료!")

## 3b단계: 메모리 검색 도구 추가

에이전트가 장기 메모리를 검색할 수 있는 도구를 생성합니다:

In [ ]:
def create_memory_retrieval_tool(memory_id: str, actor_id: str, region: str):
    """Create a tool for the agent to search its own long-term memory"""
    
    def search_long_term_memory(query: str) -> str:
        """Search long-term memory for relevant legal information about cases, precedents, strategies, and outcomes.
        
        Use this tool when you need to recall:
        - Case information (precedents, strategies, outcomes)
        - Legal precedents and their applications
        - Regulatory changes and their impacts
        - Legal strategies and their effectiveness
        - Case outcomes and lessons learned
        
        Args:
            query: Search query describing what information you need (e.g., 'CASE-001 precedents', 'contract strategies', 'Q1 outcomes')
        
        Returns:
            Relevant information from long-term memory
        """
        try:
            from bedrock_agentcore.memory.session import MemorySessionManager
            
            # 세션 매니저 생성 (memory_id와 region만 필요)
            session_manager = MemorySessionManager(
                memory_id=memory_id,
                region_name=region
            )
            
            # 시맨틱 전략 네임스페이스에서 장기 메모리 검색
            results = session_manager.search_long_term_memories(
                query=query,
                namespace_prefix="/strategies/",  # 시맨틱 전략 네임스페이스에서 검색
                top_k=5,
                max_results=10
            )
            
            if not results:
                return "장기 메모리에서 관련 정보를 찾지 못했습니다. 새로운 정보이거나 메모리 추출이 아직 처리 중일 수 있습니다."
            
            # 에이전트를 위한 결과 포맷팅
            output = "장기 메모리에서 검색된 정보:\\n\\n"
            for i, result in enumerate(results, 1):
                # MemoryRecord 객체 - content 속성에 접근
                content = getattr(result, 'content', str(result))
                # 매우 긴 콘텐츠 잘라내기
                if len(content) > 300:
                    content = content[:300] + "..."
                output += f"{i}. {content}\\n\\n"
            
            return output
            
        except Exception as e:
            return f"메모리 검색 오류: {str(e)}. 과거 컨텍스트 없이 진행합니다."
    
    return FunctionTool.from_defaults(fn=search_long_term_memory)

# 메모리 검색 도구 생성
memory_search_tool = create_memory_retrieval_tool(memory_id, "legal-analyst", region)

# 도구 목록에 메모리 검색 추가
legal_tools_with_memory = legal_tools + [memory_search_tool]

print(f"메모리 검색 도구 생성 완료! 총 도구 수: {len(legal_tools_with_memory)}")
print("   사용 네임스페이스: /strategies/ (시맨틱 전략 호환용)")

## 3c단계: 메모리 구성 확인

시맨틱 전략이 올바르게 구성되었는지 확인합니다:

In [ ]:
# 메모리 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"전략: {memory_info.get('strategies')}")
print(f"상태: {memory_info.get('status')}")
print(f"이름: {memory_info.get('name')}")

# 전략 세부 정보 표시
strategies = memory_info.get('strategies', [])
for strategy in strategies:
    print(f"\n전략 세부 정보:")
    print(f"  이름: {strategy.get('name')}")
    print(f"  유형: {strategy.get('type')}")
    print(f"  상태: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 3c단계: 메모리 구성 확인

시맨틱 전략이 올바르게 구성되었는지 확인합니다:

In [ ]:
# 메모리 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"전략: {memory_info.get('strategies')}")
print(f"상태: {memory_info.get('status')}")
print(f"이름: {memory_info.get('name')}")

# 전략 세부 정보 표시
strategies = memory_info.get('strategies', [])
for strategy in strategies:
    print(f"\n전략 세부 정보:")
    print(f"  이름: {strategy.get('name')}")
    print(f"  유형: {strategy.get('type')}")
    print(f"  상태: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 4단계: 멀티 세션 에이전트 구현

다양한 법률 기간을 시뮬레이션하기 위한 헬퍼 함수를 생성합니다:

In [ ]:
# 장기 메모리(크로스 세션) 구성
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
ANALYST_ID = "legal-analyst"  # 모든 세션에서 동일한 분석가

def create_legal_session(session_name: str):
    """장기 메모리 지속성을 갖춘 새로운 법률 세션 생성"""
    context = AgentCoreMemoryContext(
        actor_id=ANALYST_ID,         # 동일한 분석가
        memory_id=memory_id,         # 동일한 메모리 저장소 (장기 메모리 활성화)
        session_id=f"legal-{session_name}", # 기간별 다른 세션
        namespace="/legal-analysis/"
    )
    
    memory = AgentCoreMemory(context=context)
    llm = BedrockConverse(model=MODEL_ID)
    agent = FunctionAgent(
        tools=legal_tools_with_memory,  # 메모리 검색 기능이 포함된 도구 사용
        llm=llm, 
        verbose=True,  # 메모리 검색 시점을 확인하기 위해 verbose 활성화
        system_prompt="""You are a senior legal analyst with access to long-term memory.
        
CRITICAL: When asked about cases, precedents, strategies, or historical information, 
you MUST use the search_long_term_memory tool FIRST before responding.

For example:
- "What cases am I analyzing?" → Use search_long_term_memory("cases precedents")
- "What strategies have I used?" → Use search_long_term_memory("legal strategies")
- "What outcomes have I achieved?" → Use search_long_term_memory("case outcomes")

Always provide conclusive, complete responses without asking follow-up questions.\n
Execute all requested actions immediately and completely. Provide detailed, professional responses."""
    )
    
    return agent, memory

print("멀티 세션 법률 문서 분석기 설정 완료!")

## 5단계: Q1 법률 세션 - 초기 사건 분석

첫 번째 법률 세션을 시작하고 사건 기준선을 설정합니다:

In [ ]:
# === Q1 법률 세션 ===
print("=== Q1: 초기 사건 분석 ===")

agent_q1, memory_q1 = create_legal_session("q1")

# 초기 계약 조항 분석
# (시니어 법률 분석가 Sarah Chen으로서, CASE-001의 '면책' 조항 분석 요청. 리스크 수준 '높음', 판례 참조 'Smith v. Johnson (2019) - 넓은 면책 범위가 고객에게 상당한 책임 노출을 야기')
response = await agent_q1.run(
    "I'm Senior Legal Analyst Sarah Chen. Analyze contract clause for 'CASE-001' with clause type 'Indemnification', "
    "risk level 'High', precedent reference 'Smith v. Johnson (2019) - broad indemnification scope creates "
    "significant liability exposure for client'.",
    memory=memory_q1
)

print("Q1 초기 분석:")
print(response)

In [ ]:
# 초기 법률 전략 기록
# (CASE-001의 법률 전략 업데이트: 전략 유형 '계약 협상', 접근 방식 '면책 범위 축소, 중과실 및 고의적 위법 행위에 대한 예외 조항 추가', 높은 확신도)
response = await agent_q1.run(
    "Update legal strategy for 'CASE-001': strategy type 'Contract Negotiation', "
    "approach 'narrow indemnification scope, add carve-outs for gross negligence and willful misconduct', confidence level 'high'.",
    memory=memory_q1
)
print("Q1 계약 전략:", response)

# (CASE-001의 법률 전략 업데이트: 전략 유형 '리스크 완화', 접근 방식 '보험 요구사항, 책임 상한, 상호 면책 구조', 중간 확신도)
response = await agent_q1.run(
    "Update legal strategy for 'CASE-001': strategy type 'Risk Mitigation', "
    "approach 'insurance requirements, liability caps, and mutual indemnification structure', confidence level 'medium'.",
    memory=memory_q1
)
print("Q1 리스크 전략:", response)

In [ ]:
# 이벤트가 저장되었는지 확인
print("\n이벤트 저장 확인 중...")
try:
    from bedrock_agentcore.memory import MemoryClient
    client = MemoryClient(region_name=region)
    # 메모리 컨텍스트에서 session_id 가져오기
    current_session_id = memory_q1.context.session_id
    events = client.list_events(
        memory_id=memory_id,
        actor_id=ANALYST_ID,
        session_id=current_session_id  # 동적 - 현재 세션 사용
    )
    print(f"{current_session_id}에 {len(events)}개의 대화 이벤트가 저장되었습니다")
except Exception as e:
    print(f"경고: 이벤트 확인 불가: {e}")

# 시맨틱 메모리 처리를 위한 대기 시간
import asyncio
print("\n시맨틱 메모리 추출 및 인덱싱 대기 중...")
print("   (AgentCore가 백그라운드에서 대화 이벤트를 처리합니다)")
await asyncio.sleep(90)  # 메모리 추출을 위한 대기 시간 증가
print("메모리 처리 완료 - 이제 메모리를 검색할 수 있습니다")

## 6단계: Q2 법률 세션 - 규제 업데이트 대응

장기 메모리 검색을 테스트하고 규제 변화에 적응합니다:

In [ ]:
# === Q2 법률 세션 ===
print("\n=== Q2: 규제 업데이트 대응 (새 세션) ===")

agent_q2, memory_q2 = create_legal_session("q2")

# 크로스 세션 사건 회상 테스트 - 에이전트가 search_long_term_memory 도구를 사용해야 함
print("\n세션 간 메모리 검색 테스트 중...")
print("   (에이전트가 search_long_term_memory 도구를 사용하는지 확인하세요)\n")

# (분석 중인 사건이 무엇인지, 리스크 수준, 전략, 판례는 무엇인지 질문)
response = await agent_q2.run(
    "What cases am I analyzing? What are their risk levels, strategies, and precedents?",
    memory=memory_q2
)

print("\nQ2 사건 회상:")
print(response)
print("\n예상 결과: CASE-001, 면책 분석, Smith v. Johnson 판례")

In [ ]:
# 규제 변경 기록
# (규제 변경 기록: 규제 유형 '계약법 업데이트', 변경 설명 '새로운 주법이 상업 계약의 면책 범위를 제한', 영향 평가 '고객 입장에 유리, 협상 입지 강화', 고객 영향 '법적 근거를 가지고 더 좁은 면책 조건을 밀어붙일 수 있음')
response = await agent_q2.run(
    "Document regulatory change: regulation type 'Contract Law Update', "
    "change description 'New state legislation limits indemnification scope in commercial contracts', "
    "impact assessment 'favorable for our client position, strengthens negotiation stance', "
    "client implications 'can push for narrower indemnification terms with legal backing'.",
    memory=memory_q2
)
print("Q2 규제 업데이트:", response)

# 규제 변경에 기반한 전략 업데이트
# (CASE-001의 법률 전략 업데이트: 전략 유형 '규제 레버리지', 접근 방식 '좁은 면책 입장을 지지하기 위해 새 주법을 인용, 협상 레버리지 강화', 높은 확신도)
response = await agent_q2.run(
    "Update legal strategy for 'CASE-001': strategy type 'Regulatory Leverage', "
    "approach 'cite new state legislation to support narrow indemnification position, strengthen negotiation leverage', confidence level 'high'.",
    memory=memory_q2
)
print("Q2 전략 업데이트:", response)

In [ ]:
# Q2 사건 진행 추적
# (CASE-001의 법률 지표 추적: 지표 유형 '협상 진행', 값 '유리한 조건 확보', Q2 2024)
response = await agent_q2.run(
    "Track legal metrics for 'CASE-001': metric type 'Negotiation Progress', value 'Favorable terms secured', "
    "case_id 'CASE-001', quarter 'Q2 2024'.",
    memory=memory_q2
)
print("Q2 진행:", response)

# 전략 비교 테스트
# (규제 변경이 CASE-001의 전략에 어떤 영향을 미쳤는지, Q1 vs Q2 접근 방식 비교 질문)
response = await agent_q2.run(
    "How did the regulatory change impact CASE-001's strategy? Compare Q1 vs Q2 approaches.",
    memory=memory_q2
)
print("Q2 전략 분석:")
print(response)
print("\n예상 결과: Q1 계약 협상 → Q2 규제 레버리지, 입지 강화")

## 7단계: Q3 법률 세션 - 사건 해결 및 새로운 사안

사건 해결로 진행하고 새로운 사건을 접수합니다:

In [ ]:
# === Q3 법률 세션 ===
print("\n=== Q3: 사건 해결 및 새로운 사안 ===")

agent_q3, memory_q3 = create_legal_session("q3")

# 사건 결과 기록
# (CASE-001의 사건 결과 기록: 결과 유형 '합의', 결과 '유리한 조건 달성 - 좁은 면책 범위, $500K 책임 상한, 상호 구조', 교훈 '규제 레버리지가 결정적이었음, 초기 판례 조사가 성과를 거둠, 고객에게 약 $2M의 잠재적 책임을 절감')
response = await agent_q3.run(
    "Log case outcome for 'CASE-001' with outcome type 'Settlement Agreement', "
    "result 'Favorable terms achieved - narrow indemnification scope, liability caps at $500K, mutual structure', "
    "lessons learned 'regulatory leverage was decisive, early precedent research paid off, client saved estimated $2M in potential liability'.",
    memory=memory_q3
)
print("Q3 사건 해결:", response)

# 새로운 사건 분석 시작
# (CASE-002의 계약 조항 분석: 조항 유형 '경업금지', 리스크 수준 '중간', 판례 참조 'TechCorp v. StartupInc (2020) - 지리적 및 시간적 범위가 합리적이어야 함')
response = await agent_q3.run(
    "Analyze contract clause for 'CASE-002': clause type 'Non-Compete', "
    "risk level 'Medium', precedent reference 'TechCorp v. StartupInc (2020) - geographic and temporal scope must be reasonable'.",
    memory=memory_q3
)
print("Q3 새 사건 분석:", response)

In [ ]:
# 포괄적인 법률 이력 회상 테스트
# (전체 법률 분석 이력 요청: 모든 사건, 전략, 규제 변경, 결과 포함)
response = await agent_q3.run(
    "What is the complete legal analysis history? Include all cases, strategies, "
    "regulatory changes, and outcomes.",
    memory=memory_q3
)
print("Q3 전체 이력:")
print(response)
print("\n예상 결과: CASE-001 여정 → CASE-002 시작, 규제 업데이트, 전략 진화")
# 주요 법률 발견 사항을 명시적으로 추적
# (법률 발견 사항 저장: '계약에 3개의 고위험 조항 포함', 확신도 '높음')
await agent_q3.run(
    "Save legal finding: finding 'Contract contains 3 high-risk clauses', confidence 'high'.",
    memory=memory_q3
)

In [ ]:
# 시맨틱 메모리 처리를 위한 대기 시간
import asyncio
print("\n법률 메모리 추출 대기 중...")
await asyncio.sleep(90)
print("법률 메모리 처리 완료")

## 8단계: Q4 법률 세션 - 연말 리뷰 및 계획

시맨틱 검색과 연간 법률 분석을 테스트합니다:

In [ ]:
# === Q4 법률 세션 ===
print("\n=== Q4: 연말 리뷰 및 계획 ===")

agent_q4, memory_q4 = create_legal_session("q4")

# 연간 법률 지표 추적
# (법률 지표 추적: 지표 유형 '2024 연간 성과', 값 '해결 사건: 2건, 성공률: 100%, 고객 절감: $2.5M', ANNUAL-SUMMARY, 2024 연간)
response = await agent_q4.run(
    "Track legal metrics: metric type '2024 Annual Performance', value 'Cases resolved: 2, Success rate: 100%, Client savings: $2.5M', "
    "case_id 'ANNUAL-SUMMARY', quarter '2024 Annual'.",
    memory=memory_q4
)
print("Q4 연간 지표:", response)

# 규제 영향 상관관계 테스트
# (올해 기록한 규제 변경이 무엇이었는지, 사건 전략에 어떤 영향을 미쳤는지 질문)
response = await agent_q4.run(
    "What regulatory changes have I documented this year? How did they impact case strategies?",
    memory=memory_q4
)
print("Q4 규제 영향 분석:")
print(response)
print("\n예상 결과: 계약법 업데이트 → CASE-001 협상 입지 강화")

In [ ]:
# 유사한 법률 전략에 대한 시맨틱 검색 테스트
# (사용한 법률 전략이 무엇이었는지, 사건 결과에 기반하여 가장 효과적이었던 것은 무엇인지 질문)
response = await agent_q4.run(
    "What legal strategies have I used? Which were most effective based on case outcomes?",
    memory=memory_q4
)
print("Q4 전략 효과 분석:")
print(response)
print("\n예상 결과: 계약 협상 + 규제 레버리지 = 성공적 결과")

## 9단계: 2년차 Q1 세션 - 다년 법률 관점

장기 법률 지식과 업무 진화를 테스트합니다:

In [ ]:
# === 2년차 Q1 법률 세션 ===
print("\n=== 2년차 Q1: 다년 법률 관점 ===")

agent_y2q1, memory_y2q1 = create_legal_session("year2-q1")

# 다년 법률 업무 분석
# (법률 업무 진화 분석 요청: 지난 1년간 사건과 전략이 어떻게 발전했는지, 주요 법률적 결정과 그 결과는 무엇이었는지)
response = await agent_y2q1.run(
    "Analyze my legal practice evolution: How have my cases and strategies developed over the past year? "
    "What were the key legal decisions and their outcomes?",
    memory=memory_y2q1
)
print("2년차 Q1 업무 분석:")
print(response)
print("\n예상 결과: CASE-001 → CASE-002 진행, 규제 적응, 전략 정교화")

In [ ]:
# 법률 판례 진화 추적 테스트
# (법률 판례와 전략이 어떻게 진화했는지, 어떤 판례법을 적용했고 그 이유는 무엇인지 질문)
response = await agent_y2q1.run(
    "How have my legal precedents and strategies evolved? What case law have I applied and why?",
    memory=memory_y2q1
)
print("2년차 Q1 판례 진화:")
print(response)
print("\n예상 결과: 면책 판례로 시작 → 경업금지 법률로 확대")

## 10단계: 최종 법률 업무 평가

장기 법률 분석 역량에 대한 포괄적 테스트:

In [ ]:
# 최종 포괄적 법률 업무 쿼리
# (전체 법률 업무 포트폴리오 제공 요청: 모든 사건의 법적 여정, 전략 효과, 적용된 규제 변경, 판례 활용, 사건 결과 포함. 교훈과 개발된 베스트 프랙티스도 포함)
response = await agent_y2q1.run(
    "Provide my complete legal practice portfolio: all cases with their legal journeys, "
    "strategy effectiveness, regulatory changes applied, precedent utilization, and case outcomes. "
    "Include lessons learned and best practices developed.",
    memory=memory_y2q1
)
print("전체 법률 업무 포트폴리오:")
print(response)
print("\n예상 결과: 전체 사건 포트폴리오 - 전략 진화, 규제 적응, 결과 분석 포함")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 이전에 논의된 정보를 회상할 수 있는지 확인"""
        has_content = len(response) > 50
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동하고 있습니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하지만, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 심각한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - 에이전트가 논의된 내용을 회상할 수 있는지 확인
# (이 세션에서 지금까지 무엇을 논의했는지 질문)
response1 = await agent_y2q1.run("What have we discussed so far in this session?", memory=memory_y2q1)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리 - 에이전트가 컨텍스트를 유지하는지 확인
# (이전에 무엇에 대해 이야기했는지 질문)
response2 = await agent_y2q1.run("What did we talk about earlier?", memory=memory_y2q1)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 능력 - 이전 컨텍스트와 연결할 수 있는지 확인
# (이것이 이전에 논의한 내용과 어떻게 관련되는지 질문)
response3 = await agent_y2q1.run("How does this relate to what we discussed before?", memory=memory_y2q1)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 노트북에서 다음을 시연했습니다:

**장기 메모리 통합**: 크로스 세션 법률 분석을 위한 AgentCore 메모리와 LlamaIndex 활용

**법률 사건 추적**: 여러 분기에 걸친 사건 진화 및 전략 개발

**규제 인텔리전스**: 규제 변경과 사건 적용의 시맨틱 검색

**법률 전략 진화**: 초기 분석에서 규제 적응형 접근 방식으로의 자연스러운 발전

**판례 관리**: 판례법과 전략적 적용에 대한 상세 추적

**법률 업무 우수성**: 시간에 걸친 포괄적 사건 관리 및 결과 최적화

법률 문서 분석기는 장기 메모리가 어떻게 분석기를 시간이 지남에 따라 더 현명해지는 지속적 법률 파트너로 변환하는지 보여줍니다. 완전한 사건 이력을 유지하고 장기 업무 기간 전반에 걸쳐 정교한 법률 지식 검색을 가능하게 합니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다:

**참고**: 메모리를 영구적으로 삭제하려는 경우에만 실행하세요. memory_id 변수에는 이 노트북에서 앞서 생성한 메모리의 ID가 포함되어 있어야 합니다.

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    from bedrock_agentcore.memory import MemoryClient
    
    client = MemoryClient(region_name=region)
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
    
except NameError as e:
    print(f"경고: 변수가 정의되지 않았습니다: {e}")
    print("노트북을 처음부터 실행하거나 변수를 수동으로 설정하세요:")
    print("# memory_id = 'your-memory-id-here'")
    print("# region = 'us-east-1'")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")